In [1]:
# Importy
import pandas as pd
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.datasets import DenoisingAutoEncoderDataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
import os

# Folder z wieloma plikami CSV
folder_path = "../data/tsdae_dataset/" 
all_dfs = []

# Połączenie danych z każdego CSV
for filename in os.listdir(folder_path):
    if filename.endswith(".csv"):
        filepath = os.path.join(folder_path, filename)
        df = pd.read_csv(filepath)
        df = df.dropna(subset=["Text"])
        all_dfs.append(df)

# Jeden wspólny DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)
texts = combined_df["Text"].astype(str).tolist()

print(f"Wczytano {len(texts)} rekordów z {len(all_dfs)} plików.")
             
# Model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = SentenceTransformer(model_name)

# Dataset TSDAE
dataset = DenoisingAutoEncoderDataset(sentences=texts)

# DataLoader
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Loss
loss = losses.DenoisingAutoEncoderLoss(model)

# Trening
model.fit(
    train_objectives=[(dataloader, loss)],
    epochs=4,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="../models/parsed_tsdae_model_v02"  
)

C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wczytano 17087 rekordów z 2 plików.


Some weights of BertLMHeadModel were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.laye

Step,Training Loss
500,6.443200
1000,2.606300
1500,1.686500
2000,1.327700
2500,1.158100
3000,1.060300
3500,0.986600
4000,0.953500
